# 01. Exploratory Data Analysis (EDA) - Smart Energy Monitoring

**Objective:** Analyze the UCI Appliances Energy Prediction dataset (10-minute intervals) to examine distribution characteristics, environmental correlations, diurnal patterns, and time-series autocorrelation dynamics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('../data/raw/energydata_complete.csv')
df['date'] = pd.to_datetime(df['date'])
print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 1. Target Variable Distribution: `Appliances` (Wh)

Investigating skewness and base load vs. peak appliance consumption.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['Appliances'], bins=60, kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Appliances Consumption Distribution (Linear Scale)')
axes[0].set_xlabel('Energy (Wh)')

sns.histplot(np.log1p(df['Appliances']), bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Log1p(Appliances) Distribution')
axes[1].set_xlabel('log(Energy + 1)')

plt.tight_layout()
plt.show()

print("Descriptive Statistics for Appliances (Wh):")
print(df['Appliances'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

## 2. Temporal & Diurnal Energy Patterns

Aggregating energy demand by hour of the day and day of the week to detect usage cycles.

In [ ]:
df['hour'] = df['date'].dt.hour
df['day_name'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.weekday.isin([5, 6]).astype(int)

hourly_summary = df.groupby(['hour', 'is_weekend'])['Appliances'].mean().reset_index()
hourly_summary['Day Type'] = hourly_summary['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

plt.figure(figsize=(12, 5))
sns.lineplot(data=hourly_summary, x='hour', y='Appliances', hue='Day Type', marker='o')
plt.title('Average Hourly Appliance Energy Consumption: Weekday vs Weekend')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Mean Energy Consumption (Wh)')
plt.xticks(range(0, 24))
plt.show()

## 3. Environmental Temperature & Humidity Heatmap

Evaluating indoor/outdoor sensor cross-correlations (T1-T9, RH1-RH9) to identify feature collinearity.

In [ ]:
temp_cols = [c for c in df.columns if c.startswith('T') and c != 'T_out']
rh_cols = [c for c in df.columns if c.startswith('RH_')]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(df[temp_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], cbar=False)
axes[0].set_title('Indoor Temperature Correlation Matrix (T1 - T9)')

sns.heatmap(df[rh_cols].corr(), annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1], cbar=False)
axes[1].set_title('Indoor Humidity Correlation Matrix (RH1 - RH9)')

plt.tight_layout()
plt.show()

## 4. Autocorrelation & Modeling Baseline Dynamics

Analysis of lag autocorrelation demonstrating why persistence (y_t approx y_t-1) is a strong baseline at 10-minute intervals.

In [ ]:
lags = [1, 2, 3, 6, 12, 24, 144]
autocorrs = {f'Lag {l} ({l*10}m)': df['Appliances'].autocorr(lag=l) for l in lags}
autocorr_df = pd.DataFrame(list(autocorrs.items()), columns=['Lag Interval', 'Autocorrelation'])

plt.figure(figsize=(10, 4))
sns.barplot(data=autocorr_df, x='Lag Interval', y='Autocorrelation', palette='Blues_d')
plt.title('Autocorrelation of Appliance Consumption across Lags')
plt.ylabel('Pearson r')
plt.ylim(0, 1.0)
for index, row in autocorr_df.iterrows():
    plt.text(index, row['Autocorrelation'] + 0.02, f"{row['Autocorrelation']:.3f}", ha='center')
plt.show()